In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from pathlib import Path

In [ ]:

# URLs de Inside Airbnb para Madrid
URL_LISTINGS = "https://data.insideairbnb.com/spain/comunidad-de-madrid/madrid/2025-09-14/visualisations/listings.csv"
URL_LISTINGS_DEV = "https://data.insideairbnb.com/spain/comunidad-de-madrid/madrid/2025-09-14/data/listings.csv.gz"

# Importación de datos
listings = pd.read_csv(URL_LISTINGS)
listings_det = pd.read_csv(URL_LISTINGS_DEV, compression="gzip")


In [ ]:
listings.head()

In [ ]:
listings.columns

In [ ]:
a_eliminar = [ 'host_name', 'host_id', 'number_of_reviews', 'last_review',
       'reviews_per_month', 'calculated_host_listings_count', 'number_of_reviews_ltm', 'license' ]

# Eliminamos las columnas innecesarias
listings = listings.drop(columns=a_eliminar)

In [ ]:
listings.head()

In [ ]:
listings.info()

In [ ]:
# analisis de nulos
listings.isnull().sum()

In [ ]:
# eliminamos filas con nulos
listings = listings.dropna()

In [ ]:
# verificamos que no hay filas duplicadas
listings.duplicated().sum()

In [ ]:
# Value counts de las variables categóricas
cols_object = listings.select_dtypes(include="object").columns

for col in cols_object:
    print(f"\n{'='*50}")
    print(f"Variable: {col}")
    print(f"{'='*50}")
    print(listings[col].value_counts(dropna=False))


In [ ]:
# eliminar los registros ddnde room_type sea igual a Hotel room
listings = listings[listings["room_type"] != "Hotel room"]

In [ ]:
listings.iloc[:, 7:10].describe().T

In [ ]:
listings_filtrado = listings[listings["price"] > 1000]
listings_filtrado["price"].plot(kind="hist", bins=20, title="Distribución de precios > 1000€")


In [ ]:
# quedarnos con filas cuyo precio sea mayor a 20 y menor a 1000 
listings = listings[(listings["price"] >= 20) & (listings["price"] <= 1000)]
listings["price"].plot(kind="hist", bins=20, title="Distribución de precios entre 20 y 1000")

In [ ]:
listings.iloc[:, 7:10].describe().T

In [ ]:
listings_det.head(6)

In [ ]:
listings_det.property_type.value_counts()

In [ ]:
listings_det.info()

In [ ]:
a_incluir = [
    'id',
    'description',
    'accommodates',
    'bathrooms',
    'bedrooms',
    'beds',
    'review_scores_location',
    'estimated_occupancy_l365d',
]

listings_det = listings_det[a_incluir]

In [ ]:
listings_det.info()

In [ ]:
# analisis de nulos
listings_det.isnull().sum()

In [ ]:
listings_det.head(5) 

In [ ]:
# numero de registros con estimated_occupancy_l365d igual a 0
listings_det[listings_det["estimated_occupancy_l365d"] == 0].shape[0]

In [ ]:
# eliminar los registros con estimated_occupancy_l365d igual a 0
listings_det = listings_det[listings_det["estimated_occupancy_l365d"] != 0]

In [ ]:
listings_det.isnull().sum()

In [ ]:
listings_det.shape

In [ ]:
listings_det.accommodates.value_counts(ascending=False)

In [ ]:
# eliminar registros con accommodates mayores que 8 
listings_det = listings_det[listings_det["accommodates"] <= 8]

In [ ]:
listings_det.accommodates.value_counts(ascending=False)

In [ ]:
# eliminar registros con bathrooms mayores que 5
listings_det = listings_det[listings_det["bathrooms"] <= 5]

In [ ]:
listings_det.bathrooms.value_counts(ascending=False)

In [ ]:
# eliminar registros con bedrooms mayores que 5
listings_det = listings_det[listings_det["bedrooms"] <= 5]

In [ ]:
listings_det.bedrooms.value_counts(ascending=False)

In [ ]:
# eliminar registros con beds mayores que 6
listings_det = listings_det[listings_det["beds"] <= 6]

In [ ]:
listings_det.beds.value_counts(ascending=False)


In [ ]:
listings_det.isnull().sum()

In [ ]:
listings_det.shape

In [ ]:
# crosstab entre accommodates y bedrooms                      TECNICA PARA LA CREACION DE UN PROXY
pd.crosstab(listings_det["accommodates"], listings_det["bedrooms"], margins=True)

In [ ]:
# Comprobar si existen duplicados en listings_det
listings_det.duplicated().sum()

In [ ]:
df_idealista = pd.read_csv("../datos/brutos/idealista.csv")

In [ ]:
df_idealista.columns = ["precio", "distrito"]

# Eliminar el primer registro 
df_idealista = df_idealista.iloc[1:]

df_idealista.head()

In [ ]:
df_idealista["precio"] = (
    df_idealista["precio"]
    .str.replace("€", "", regex=False)
    .str.replace("/m2", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)  # Por si hay comas decimales
    .str.strip()
)

In [ ]:
df_idealista["precio"] = df_idealista["precio"].astype(int)

In [ ]:
df_idealista.info()

In [ ]:
df_idealista

In [ ]:
distritos_l = listings.neighbourhood_group.unique()
distritos_i = df_idealista.distrito.unique()

In [ ]:
distritos_l

In [ ]:
distritos_i

In [ ]:
# Diccionario de mapeo para unificar nombres
mapeo_distritos = {
    "San Blas": "San Blas - Canillejas",
    "Fuencarral": "Fuencarral - El Pardo",
    "Moncloa": "Moncloa - Aravaca"
}

# Aplicar el mapeo en el dataframe de idealista
df_idealista["distrito"] = df_idealista["distrito"].replace(mapeo_distritos)

In [ ]:
# Realizar el merge por el nombre del distrito
df_merged = listings.merge(df_idealista, left_on="neighbourhood_group", right_on="distrito", how="left")

In [ ]:
df_merged.isnull().sum()

In [ ]:
df_merged.info()

In [ ]:
listings.shape

In [ ]:
df_merged.shape

In [ ]:
import sqlite3
import os

# Crear la ruta completa para la base de datos en la carpeta intermedios
ruta_db = os.path.join("..", "datos", "intermedios", "analisis_inmobiliario.db")

# Crear/conectar a la base de datos
conn = sqlite3.connect(ruta_db)

# Guardar los dataframes como tablas
listings.to_sql("listings", conn, if_exists="replace", index=False)
listings_det.to_sql("listings_det", conn, if_exists="replace", index=False)
df_idealista.to_sql("idealista", conn, if_exists="replace", index=False)

In [ ]:
# Comprobación: mostrar las tablas creadas
tablas = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
print("Tablas en la base de datos:", [t[0] for t in tablas])

conn.close()